# 09 — Listing-Level Price Prediction (Member 3)

**KPMG slide requirement** ('Price Prediction Modeling' + 'Evaluation and Metrics'). Not consumed by the chatbot. Logic in `src/price_model.py`.

Target `ttm_avg_rate` is modelled as `log1p` (right-skewed) and metrics are inverted to currency units. The data loader globs whatever `*_listings_clean.csv` files are present, so a full local clone (Barcelona + London) retrains the combined model with no code change.

In [1]:
import sys
from pathlib import Path
# Resolve repo root whether run from notebooks/ or repo root
_here = Path.cwd()
ROOT = _here if (_here / 'src').exists() else _here.parent
sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, joblib
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.data_io import PROCESSED_DIR
print('repo root:', ROOT)

repo root: /Users/bobbypakenham/KPMG_Airbnb_Capstone


In [2]:
from src import price_model as pm
data = pm.load_price_frame()
print('rows:', len(data), '| cities:', sorted(data.city.unique()))

rows: 8214 | cities: ['barcelona', 'london']


## Train linear / random forest / gradient boosting; select best by hold-out R²

In [3]:
res = pm.train_and_select(data)
print('best:', res['best_name'], '| train', res['n_train'], 'test', res['n_test'], '| cities', res['cities'])
res['metrics']

best: gradient_boosting | train 6571 test 1643 | cities ['barcelona', 'london']


,rmse,mae,r2,cv_r2_mean,cv_r2_std
linear,132.99,73.73,0.5644,0.6777,0.0169
random_forest,129.99,73.51,0.5838,0.6778,0.0103
gradient_boosting,126.86,71.14,0.6036,0.6988,0.0141


## Feature importance (best tree model)

In [4]:
res['feature_importance'].head(15)

,feature,importance
0,cat__room_type_entire_home,0.231285
1,boo__entire_home_flag,0.218388
2,num__guests,0.117113
3,num__baths,0.113835
4,num__bedrooms,0.080723
5,cat__neighborhood_Royal Borough of Kensington ...,0.022848
6,num__beds,0.022529
7,cat__listing_type_Room in hotel,0.017308
8,cat__listing_type_Room in boutique hotel,0.014017
9,cat__city_london,0.013659


## Save the model

In [5]:
(ROOT/'models').mkdir(exist_ok=True)
joblib.dump(res['best_pipeline'], ROOT/'models'/'price_model.joblib')
print('saved models/price_model.joblib')
# sanity: reload and predict one row
m = joblib.load(ROOT/'models'/'price_model.joblib')
X,_,_ = pm.build_xy(data)
print('sample prediction (currency):', round(float(np.expm1(m.predict(X.head(1))[0])),2))

saved models/price_model.joblib
sample prediction (currency): 79.69
